# Manual review of emotion stories

Samples stories by emotion for manual inspection, saves judgments, and summarizes label quality. Runs in Jupyter or Colab on CPU.


## 1. Load the dataset
Reads an external JSONL file and configures local review paths.


In [ ]:
import json, random, hashlib, zipfile, io, os
from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime, timezone
from IPython.display import clear_output, display, FileLink
import pandas as pd

DATA_PATH = Path("data/emotion_stories_qwen_full1120.jsonl").expanduser().resolve()
OUTPUT_ROOT = Path("outputs/story_review").expanduser().resolve()
RESTORE_ZIP_PATH = None  # Optional path to a previously exported review ZIP.
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Set DATA_PATH to your dataset. Not found: {DATA_PATH}")
blob = DATA_PATH.read_bytes()
dataset_hash=hashlib.sha256(blob).hexdigest()
records=[]
for line_number,line in enumerate(blob.decode('utf-8-sig').splitlines(),1):
    if not line.strip(): continue
    r=json.loads(line)
    if not isinstance(r,dict) or not {'id','emotion','text'} <= r.keys():
        raise ValueError(f'Line {line_number}: need id, emotion, text.')
    if any(not isinstance(r[k],str) or not r[k].strip() for k in ['id','emotion','text']):
        raise ValueError(f'Line {line_number}: invalid id, emotion, or text.')
    records.append(r)
if not records: raise ValueError('Dataset is empty.')
if len({r['id'] for r in records})!=len(records):
    raise ValueError('Duplicate story IDs.')
REVIEW_DIR=OUTPUT_ROOT/f'story_review_{dataset_hash[:12]}'
REVIEW_DIR.mkdir(parents=True,exist_ok=True)
SAMPLE_PATH=REVIEW_DIR/'sample.json'
RATINGS_PATH=REVIEW_DIR/'ratings.json'
print('Loaded stories:',len(records))
print('Emotion counts:',dict(Counter(r['emotion'] for r in records)))
print("Dataset:", DATA_PATH)
print("Review folder:", REVIEW_DIR)


## 2. Restore saved reviews
Optionally restores a review ZIP after checking its dataset fingerprint and protecting existing files.


In [ ]:
if RESTORE_ZIP_PATH is not None:
    restore_path = Path(RESTORE_ZIP_PATH).expanduser().resolve()
    with zipfile.ZipFile(restore_path) as archive:
        allowed={'sample.json','ratings.json','review_details.csv','review_summary.csv'}
        if any(x not in allowed for x in archive.namelist()):
            raise ValueError('Unexpected ZIP contents.')
        saved=json.loads(archive.read('sample.json'))
        if saved['dataset_hash']!=dataset_hash:
            raise ValueError('Review ZIP belongs to a different dataset.')
        for entry in archive.namelist():
            target=REVIEW_DIR/entry
            if target.exists() and target.read_bytes()!=archive.read(entry):
                raise ValueError(f'Different local file already exists: {entry}')
        archive.extractall(REVIEW_DIR)
    print('Restored review.')


## 3. Draw the sample
Selects up to ten stories per emotion with a fixed seed; preserves the same sample on reruns.


In [ ]:
N_PER_EMOTION=10
REVIEW_SEED=180

def choose_sample(records,n,seed):
    groups=defaultdict(list)
    for r in records: groups[r['emotion']].append(r)
    rng=random.Random(seed)
    selected=[]
    for emotion in sorted(groups):
        candidates=sorted(groups[emotion],key=lambda r:r['id'])
        selected.extend(rng.sample(candidates,min(n,len(candidates))))
    rng.shuffle(selected)
    return selected

def atomic_save(path,value):
    tmp=path.with_suffix('.tmp')
    tmp.write_text(json.dumps(value,ensure_ascii=False,indent=2),encoding='utf-8')
    os.replace(tmp,path)

if SAMPLE_PATH.exists():
    sample_info=json.loads(SAMPLE_PATH.read_text())
    if (sample_info['dataset_hash']!=dataset_hash or
        sample_info['seed']!=REVIEW_SEED or
        sample_info['n_per_emotion']!=N_PER_EMOTION):
        raise ValueError('Saved sample settings differ. Restore the original settings.')
else:
    sample_info=dict(dataset_hash=dataset_hash,seed=REVIEW_SEED,
        n_per_emotion=N_PER_EMOTION,stories=choose_sample(records,N_PER_EMOTION,REVIEW_SEED))
    atomic_save(SAMPLE_PATH,sample_info)
sample=sample_info['stories']
lookup={r['id']:r for r in records}
if any(lookup.get(r['id'])!=r for r in sample):
    raise ValueError('Saved sample does not match dataset.')
print('Sample size:',len(sample))
print('Counts:',dict(Counter(r['emotion'] for r in sample)))
if any(n<N_PER_EMOTION for n in Counter(r['emotion'] for r in sample).values()):
    print('Some emotions have fewer than 10 stories; all available were selected.')


## 4. Inspect stories
Collects a blind first impression, intended-emotion fit, explicit-word and premise checks, and notes. Saves each completed review; q pauses.


In [ ]:
ratings=json.loads(RATINGS_PATH.read_text()) if RATINGS_PATH.exists() else {}
if not set(ratings)<=set(r['id'] for r in sample):
    raise ValueError('Ratings contain IDs outside this sample.')

def ask_choice(prompt,choices):
    while True:
        answer=input(prompt).strip().lower()
        if answer=='q': return None
        if answer in choices: return answer
        print('Enter one of:',', '.join(sorted(choices)),'or q to stop.')

for position,r in enumerate(sample,1):
    if r['id'] in ratings: continue
    clear_output(wait=True)
    print(f'Sample {position}/{len(sample)} | Completed: {len(ratings)}')
    print('Type q at any prompt to stop.\n')
    print(r['text'])
    perceived=input('\nWhat emotion do you perceive? Use your own words, or unclear: ').strip()
    if perceived.lower()=='q': break
    print('\nIntended emotion:',r['emotion'])
    fit=ask_choice('Does it convey that emotion? [y=clearly, u=unclear/mixed, n=no]: ',{'y','u','n'})
    if fit is None: break
    explicit=ask_choice('Does it name that emotion or a direct synonym? [y/n]: ',{'y','n'})
    if explicit is None: break
    topic=r.get('topic')
    if topic:
        premise=ask_choice(f'Does it follow this premise: {topic}? [y/n]: ',{'y','n'})
        if premise is None: break
    else:
        premise='na'
        print('No topic text in record; premise check skipped.')
    notes=input('Optional notes (ambiguity, repetition, coherence, etc.): ').strip()
    if notes.lower()=='q': break
    ratings[r['id']]=dict(emotion=r['emotion'],perceived_emotion=perceived,
        intended_emotion_fit=fit,explicit_emotion_term=explicit,
        follows_premise=premise,notes=notes,
        reviewed_utc=datetime.now(timezone.utc).isoformat())
    atomic_save(RATINGS_PATH,ratings)
print(f'Saved {len(ratings)}/{len(sample)} reviews. Run section 5 to export progress.')


## 5. Summarize and export
Reports quality rates and flagged stories, then saves CSV files and a review ZIP. Rates use completed reviews; missing premises are not passes.


In [ ]:
ratings=json.loads(RATINGS_PATH.read_text()) if RATINGS_PATH.exists() else {}
summary_rows=[]
for emotion in sorted({r['emotion'] for r in sample}):
    selected=[r for r in sample if r['emotion']==emotion]
    judged=[ratings[r['id']] for r in selected if r['id'] in ratings]
    n=len(judged)
    rate=lambda condition: sum(condition(r) for r in judged)/n if n else None
    known=[r for r in judged if r['follows_premise']!='na']
    summary_rows.append(dict(emotion=emotion,sampled=len(selected),reviewed=n,
        clear_emotion_rate=rate(lambda r:r['intended_emotion_fit']=='y'),
        no_explicit_term_rate=rate(lambda r:r['explicit_emotion_term']=='n'),
        core_pass_rate=rate(lambda r:r['intended_emotion_fit']=='y' and r['explicit_emotion_term']=='n'),
        premise_reviewed=len(known),
        correct_premise_rate=sum(r['follows_premise']=='y' for r in known)/len(known) if known else None,
        full_pass_rate=rate(lambda r:r['intended_emotion_fit']=='y' and
            r['explicit_emotion_term']=='n' and r['follows_premise']=='y')))
summary=pd.DataFrame(summary_rows).set_index('emotion')
display(summary.round(2))
print(f'Completed: {len(ratings)}/{len(sample)}')
review=pd.DataFrame([dict(story_id=k,**v) for k,v in ratings.items()])
if not review.empty:
    attention=review[(review.intended_emotion_fit!='y') |
        (review.explicit_emotion_term!='n') | (review.follows_premise=='n')]
    print('Stories needing attention:')
    display(attention)
review.to_csv(REVIEW_DIR/'review_details.csv',index=False)
summary.to_csv(REVIEW_DIR/'review_summary.csv')
archive=REVIEW_DIR/'story_manual_review.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for name in ['sample.json','ratings.json','review_details.csv','review_summary.csv']:
        path=REVIEW_DIR/name
        if path.exists(): z.write(path,name)
print("Saved ZIP:", archive)
display(FileLink(os.path.relpath(archive, Path.cwd())))
